In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import os
import sys

root_path = os.path.abspath("../..")
sys.path.append(root_path)

from algorithm.MADDPG import MADDPGTrainer, MADDPGTester
from network_env.network_env_v7 import NetworkEnvV7, ProcessorSharingScheduler

### Directories

In [11]:
RESOURCE_PATH = "./configs/resource_config.json"
LOG_PATH = "./results"

### Configurations

In [12]:

frames_per_batch = 100
n_iter = 100
min_replay_size = 1000
memory_size = 10000
n_optimizer_steps = 100
train_batch_size = 128
actor_lr = 1e-4
critic_lr = 1e-4
max_grad_norm = 1.0
gamma = 0.99
polyak_tau = 0.005

critic_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter": True,
        "centralized_critic": True
    }

actor_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter":True
    }

MAX_QUEUE_LENGTH = 5 #5 times capacity
PENALTY = -5
ALPHA = 1.0
BETA = 100.0


### Experiment 1

- Single slice
- Constant Demand = 0.5
- (lambda, rho) = (0.5,0.5), (0.1,0.9), (0.9,0.1)

In [13]:
n_agent = 1
test_demand = 0.5
latency_pref = [0.5, 0.1, 0.9]
energy_pref = [0.5, 0.9, 0.1]


In [14]:
for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"1.{idx}")
    
    train_env = NetworkEnvV7(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_],
        scheduler=ProcessorSharingScheduler()
    )

    test_env = NetworkEnvV7(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_],
        scheduler=ProcessorSharingScheduler()
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

idx=0, lambda = 0.5, rho = 0.5


2026-06-15 18:32:46,544 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([10000]) shape [END]


slice_r: -4.959:   1%|          | 1/100 [02:09<3:33:49, 129.59s/it]
/spare/tngo/MARL4NetworkSlicing/algorithm/MADDPG.py:418: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  po

Saved trained policy weights for group 'slice' to ./results/1.0/maddpg_actor_slice.pt
Successfully loaded weights for group 'slice' from ./results/1.0/maddpg_actor_slice.pt
Starting Deterministic Evaluation...


100%|██████████| 10/10 [00:04<00:00,  2.10it/s]


...

/home/tngo/MARL4NetworkSlicing/.venv/lib/python3.12/site-packages/torchrl/envs/libs/pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


[Recorder.save_result] Saved 9994 rewards, 9994 latencies, 9994 energies to results/1.1/train/slice_0
[save_statistics] Saving recorder statistics...
[Recorder.save_result] Saved 990 rewards, 990 latencies, 990 energies to results/1.1/test/slice_0
idx=2, lambda = 0.9, rho = 0.1


2026-06-15 18:37:14,819 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([10000]) shape [END]


  0%|          | 0/100 [06:33<?, ?it/s]


KeyboardInterrupt: 

### Experiment 2

- Single slice
- Constant Demand = 0.3,0.5,0.7
- (lambda, rho) = (0.5,0.5)

In [ ]:
n_agent = 1
test_demand = [0.3,0.5,0.7]
latency_pref = 0.5
energy_pref = 0.5

In [ ]:
for idx, demand_ in enumerate(test_demand):
    print(f'idx={idx}, demand = {demand_}')
    
    log_path = os.path.join(LOG_PATH,f"2.{idx}")
    
    train_env = NetworkEnvV7(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref],
        scheduler=ProcessorSharingScheduler()
    )

    test_env = NetworkEnvV7(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref],
        scheduler=ProcessorSharingScheduler()
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

/home/tngo/MARL4NetworkSlicing/.venv/lib/python3.12/site-packages/torchrl/envs/libs/pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


idx=0, demand = 0.3


  0%|          | 0/100 [00:00<?, ?it/s]

2026-06-15 18:31:07,225 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([10000]) shape [END]


KeyboardInterrupt: 